# Excelファイルから画像を抽出するサンプルコード

Excelファイル（`.xlsx`）に含まれるすべての画像を抽出する2つの方法を紹介します。

1. **標準ライブラリ `zipfile` を使用する方法**（シンプル・高速・追加パッケージ不要）
2. **`openpyxl` を使用する方法**（シート名やセルの位置情報などを確認しながら抽出したい場合）

## 準備

`openpyxl` を使用する場合は、事前に必要なパッケージをインストールしてください。
※ ユーザー指定により、パッケージ管理には `uv` を使用します。

```bash
uv pip install openpyxl Pillow
```

### 方法1: `zipfile` を使用する（最も簡単・高速）

Excelファイル（`.xlsx`）は、実態は複数のXMLファイルやリソースがまとめられたZIPアーカイブです。そのため、標準の `zipfile` モジュールを使って解凍し、画像が格納されている `xl/media/` ディレクトリから直接画像ファイルを取り出すことができます。

In [ ]:
import zipfile
import os

def extract_images_zip(excel_path, output_dir):
    """
    ExcelファイルからZIP解凍機能を使ってすべての画像を一括抽出します。
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    extracted_count = 0
    with zipfile.ZipFile(excel_path, 'r') as archive:
        for file_info in archive.infolist():
            # Excel内の画像は通常 xl/media フォルダに格納されています
            if file_info.filename.startswith('xl/media/'):
                img_filename = os.path.basename(file_info.filename)
                if img_filename:
                    dest_path = os.path.join(output_dir, img_filename)
                    # 画像データを読み込んで保存
                    with open(dest_path, 'wb') as f:
                        f.write(archive.read(file_info.filename))
                    print(f"抽出完了: {img_filename} -> {dest_path}")
                    extracted_count += 1
                    
    print(f"合計 {extracted_count} 個の画像を抽出しました。")

# 実行例
# excel_file = "example.xlsx"
# output_folder = "extracted_images_zip"
# extract_images_zip(excel_file, output_folder)

### 方法2: `openpyxl` を使用する（シート名や位置情報を取得したい場合）

`openpyxl` を使うと、どのシートのどのセル付近に画像が配置されているかなどの情報にアクセスできます。画像自体を保存する際には、`openpyxl` の画像オブジェクトからバイナリデータを取り出します。

In [ ]:
import os
from io import BytesIO
from openpyxl import load_workbook
from PIL import Image

def extract_images_openpyxl(excel_path, output_dir):
    """
    openpyxlを使用して、各シートに貼り付けられている画像を抽出します。
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ワークブックの読み込み
    wb = load_workbook(excel_path)
    extracted_count = 0
    
    for sheet in wb.worksheets:
        # シート内の画像をループ
        # ※ openpyxlでは画像は ws._images リストに格納されています
        if hasattr(sheet, '_images') and sheet._images:
            print(f"シート '{sheet.title}' から画像を抽出中...")
            for i, image in enumerate(sheet._images):
                # 画像データの取得
                img_data = image._data()
                
                # PILを使用して画像を開く
                img = Image.open(BytesIO(img_data))
                
                # 拡張子の決定（画像フォーマットから判定）
                ext = img.format.lower() if img.format else 'png'
                
                # 画像の配置位置（アンカー情報）があればファイル名に含める
                # 例: image.anchor.to.col や row、または単に A1 などの番地
                cell_ref = ""
                if hasattr(image, 'anchor') and image.anchor:
                    # 簡易的な位置情報取得（セルのアドレスなど）
                    cell_ref = f"_{image.anchor}"
                
                filename = f"{sheet.title}_img_{i+1}{cell_ref}.{ext}"
                # 不適切な文字の置換（ファイル名用）
                filename = filename.replace("'", "").replace(" ", "_")
                
                dest_path = os.path.join(output_dir, filename)
                img.save(dest_path)
                print(f"  -> {filename} を保存しました。")
                extracted_count += 1
        else:
            print(f"シート '{sheet.title}' には画像がありません。")
            
    print(f"合計 {extracted_count} 個の画像を抽出しました。")

# 実行例
# excel_file = "example.xlsx"
# output_folder = "extracted_images_openpyxl"
# extract_images_openpyxl(excel_file, output_folder)